In [ ]:
from glob import glob
import os
import pandas as pd
import h5py
import numpy as np
import openslide
from PIL import Image
import matplotlib.pyplot as plt
import anndata as ad
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)
        
organ=['Ovary', 'Breast', 'Pancreas', 'Lymphoid', 'Prostate', 'Kidney',
       'Lung', 'Skin', 'Bowel', 'Heart', 'Brain', 'Bone', 'Liver',
       'Cervix', 'Lymph node', 'Bladder', 'Eye', 'Uterus']
st_technology=['Visium HD', "Visium HD 3'", 'Xenium', 'Visium',
       'Spatial Transcriptomics']
for o in organ:
    for t in st_technology:
        create_dir(f'../../data/marker_gene_spatial_expression/label/{t}/{o}')
        create_dir(f'../../data/marker_gene_spatial_expression/image/{t}/{o}')
        create_dir(f'../../data/marker_gene_spatial_expression/image_5x/{t}/{o}')
        
mpp=20
image_size=512

In [ ]:
data_path='../../data/spatialTranscriptome/'
meta_df = pd.read_csv(f'{data_path}meta_df_homo_sapiens.csv')

slide=openslide.open_slide(f'{data_path}wsis/{meta_df.loc[51]["image_filename"]}')
# h5ad 파일 읽기
adata = ad.read_h5ad(f'{data_path}st/{meta_df.loc[51]["id"]}.h5ad')

In [ ]:
from scipy.interpolate import griddata

# marker gene 리스트
marker_genes = [
    'EPCAM', 'KRT8', 'KRT18', 'KRT19',
    'COL1A1', 'COL1A2', 'FAP', 'ACTA2',
    'CD3D', 'CD3E', 'CD8A',
    'MS4A1', 'CD79A',
    'CD68',
    'PECAM1'
]

# 존재하는 marker gene 확인
gene_names = adata.var_names.tolist()
available_genes = [g for g in marker_genes if g in gene_names]
missing_genes = [g for g in marker_genes if g not in gene_names]
print(f'Available: {available_genes}')
print(f'Missing: {missing_genes}')

# spatial 좌표
coords = adata.obsm['spatial']  # (n_spots, 2) -> (col, row) = (x, y)

# WSI 썸네일
slide_w, slide_h = slide.dimensions
thumb_size = 1000
thumbnail = slide.get_thumbnail((thumb_size, thumb_size))
thumb_w, thumb_h = thumbnail.size

# 좌표를 썸네일 스케일로 변환
scale_x = thumb_w / slide_w
scale_y = thumb_h / slide_h
coords_thumb = coords.copy().astype(float)
coords_thumb[:, 0] *= scale_x  # x
coords_thumb[:, 1] *= scale_y  # y

# heatmap 보간용 그리드
grid_y, grid_x = np.mgrid[0:thumb_h:128j, 0:thumb_w:128j]

# 15개 marker gene heatmap overlay
fig, axes = plt.subplots(3, 5, figsize=(25, 15))
axes = axes.flatten()

for i, gene in enumerate(marker_genes):
    ax = axes[i]
    ax.imshow(thumbnail)

    if gene in available_genes:
        gene_idx = gene_names.index(gene)
        if hasattr(adata.X, 'toarray'):
            expr = adata.X[:, gene_idx].toarray().flatten()
        else:
            expr = np.array(adata.X[:, gene_idx]).flatten()

        # (x, y) 좌표 기반 보간
        heatmap = griddata(
            coords_thumb,  # (x, y)
            expr,
            (grid_x, grid_y),  # (x, y) 그리드
            method='cubic',
            fill_value=0
        )
        heatmap = np.clip(heatmap, 0, None)

        ax.imshow(
            heatmap,
            extent=[0, thumb_w, thumb_h, 0],
            cmap='hot',
            alpha=0.5,
            interpolation='bilinear'
        )
        ax.set_title(gene, fontsize=14, fontweight='bold')
    else:
        ax.set_title(f'{gene} (N/A)', fontsize=14, fontweight='bold', color='red')

    ax.axis('off')

plt.suptitle(f'{meta_df.loc[0]["id"]} - Marker Gene Expression Heatmap', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.interpolate import LinearNDInterpolator
from scipy.spatial import Delaunay
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

marker_genes = [
    'EPCAM', 'KRT8', 'KRT18', 'KRT19',
    'COL1A1', 'COL1A2', 'FAP', 'ACTA2',
    'CD3D', 'CD3E', 'CD8A',
    'MS4A1', 'CD79A',
    'CD68',
    'PECAM1'
]

target_mag = mpp  # 20 (target 20x)
context_mag = 5   # 5x context patch
patch_size = image_size  # 512
heatmap_size = 128
heatmap_ratio = patch_size // heatmap_size  # 4
min_spots = 1

save_image_dir = '../../data/marker_gene_spatial_expression/image'
save_image_5x_dir = '../../data/marker_gene_spatial_expression/image_5x'
save_label_dir = '../../data/marker_gene_spatial_expression/label'

skipped = []
processed = []

for idx in tqdm(range(len(meta_df)), desc='Processing slides'):
    row = meta_df.iloc[idx]
    sample_id = row['id']
    organ_name = row['organ']
    st_tech = row['st_technology']
    image_file = row['image_filename']

    # h5ad 로드
    h5ad_path = f'{data_path}st/{sample_id}.h5ad'
    if not os.path.exists(h5ad_path):
        skipped.append((sample_id, 'h5ad not found'))
        continue

    adata = ad.read_h5ad(h5ad_path)
    gene_names = adata.var_names.tolist()

    missing = [g for g in marker_genes if g not in gene_names]
    if len(missing) > 0:
        skipped.append((sample_id, f'missing genes: {missing}'))
        continue

    wsi_path = f'{data_path}wsis/{image_file}'
    if not os.path.exists(wsi_path):
        skipped.append((sample_id, 'WSI not found'))
        continue

    slide = openslide.open_slide(wsi_path)
    slide_w, slide_h = slide.dimensions

    native_mag = float(str(row['magnification']).replace('x', '').replace('X', ''))
    downsample_20x = native_mag / target_mag
    downsample_5x = native_mag / context_mag

    full_patch_20x = int(patch_size * downsample_20x)
    full_patch_5x = int(patch_size * downsample_5x)

    target_w = int(slide_w / downsample_20x)
    target_h = int(slide_h / downsample_20x)

    hm_w = target_w // heatmap_ratio
    hm_h = target_h // heatmap_ratio

    if 'pxl_col_in_fullres' in adata.obs.columns and 'pxl_row_in_fullres' in adata.obs.columns:
        coords_x = adata.obs['pxl_col_in_fullres'].values.astype(float)
        coords_y = adata.obs['pxl_row_in_fullres'].values.astype(float)
    else:
        coords = adata.obsm['spatial']
        coords_x = coords[:, 0].astype(float)
        coords_y = coords[:, 1].astype(float)

    coords_20x_x = coords_x / downsample_20x
    coords_20x_y = coords_y / downsample_20x

    hm_coords_x = coords_20x_x / heatmap_ratio
    hm_coords_y = coords_20x_y / heatmap_ratio

    # 15개 gene 발현값 한번에 추출
    gene_indices = [gene_names.index(g) for g in marker_genes]
    if hasattr(adata.X, 'toarray'):
        expr_matrix = adata.X[:, gene_indices].toarray()  # (n_spots, 15)
    else:
        expr_matrix = np.array(adata.X[:, gene_indices])

    # Delaunay 삼각분할 1회만 계산 → 15개 gene 재사용
    points = np.column_stack([hm_coords_x, hm_coords_y])
    tri = Delaunay(points)

    grid_y, grid_x = np.mgrid[0:hm_h, 0:hm_w]
    grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])

    wsi_heatmaps = np.zeros((15, hm_h, hm_w), dtype=np.float32)

    for gi in range(15):
        interp = LinearNDInterpolator(tri, expr_matrix[:, gi], fill_value=0)
        heatmap = interp(grid_points).reshape(hm_h, hm_w)
        wsi_heatmaps[gi] = np.clip(heatmap, 0, None).astype(np.float32)

    # gene별 min-max 정규화 (0~1)
    for gi in range(15):
        ch = wsi_heatmaps[gi]
        ch_min, ch_max = ch.min(), ch.max()
        if ch_max > ch_min:
            wsi_heatmaps[gi] = (ch - ch_min) / (ch_max - ch_min)
        else:
            wsi_heatmaps[gi] = 0.0

    # 패치 분할 및 저장
    n_patches_x = target_w // patch_size
    n_patches_y = target_h // patch_size
    patch_count = 0

    def save_patch(px, py):
        patch_x0 = px * patch_size
        patch_y0 = py * patch_size

        in_patch = (
            (coords_20x_x >= patch_x0) & (coords_20x_x < patch_x0 + patch_size) &
            (coords_20x_y >= patch_y0) & (coords_20x_y < patch_y0 + patch_size)
        )
        if in_patch.sum() < min_spots:
            return False

        hm_x = px * heatmap_size
        hm_y = py * heatmap_size
        if hm_x + heatmap_size > hm_w or hm_y + heatmap_size > hm_h:
            return False

        label_patch = wsi_heatmaps[:, hm_y:hm_y + heatmap_size, hm_x:hm_x + heatmap_size]

        # 20x 패치
        full_x = int(px * full_patch_20x)
        full_y = int(py * full_patch_20x)
        img_region = slide.read_region((full_x, full_y), 0, (full_patch_20x, full_patch_20x))
        img_patch = img_region.convert('RGB').resize((patch_size, patch_size), Image.LANCZOS)

        # 5x 패치 (동일 중심)
        center_x = full_x + full_patch_20x // 2
        center_y = full_y + full_patch_20x // 2
        half_5x = full_patch_5x // 2
        x5 = np.clip(center_x - half_5x, 0, max(0, slide_w - full_patch_5x))
        y5 = np.clip(center_y - half_5x, 0, max(0, slide_h - full_patch_5x))

        img_region_5x = slide.read_region((int(x5), int(y5)), 0, (full_patch_5x, full_patch_5x))
        img_patch_5x = img_region_5x.convert('RGB').resize((patch_size, patch_size), Image.LANCZOS)

        patch_name = f'{sample_id}_{px}_{py}'
        img_patch.save(f'{save_image_dir}/{st_tech}/{organ_name}/{patch_name}.tiff')
        img_patch_5x.save(f'{save_image_5x_dir}/{st_tech}/{organ_name}/{patch_name}.tiff')
        np.save(f'{save_label_dir}/{st_tech}/{organ_name}/{patch_name}.npy', label_patch)
        return True

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = []
        for py in range(n_patches_y):
            for px in range(n_patches_x):
                futures.append(executor.submit(save_patch, px, py))
        for f in futures:
            if f.result():
                patch_count += 1

    slide.close()
    processed.append((sample_id, patch_count))
    print(f'{sample_id} | {st_tech}/{organ_name} | {patch_count} patches saved')

print(f'\n=== Done ===')
print(f'Processed: {len(processed)} slides')
print(f'Skipped: {len(skipped)} slides')
for s_id, reason in skipped:
    print(f'  SKIP {s_id}: {reason}')